# Chapter 13 - Data Science
## Data Preparation

## 0 - Setting up the notebook

In [10]:
import json
import random
from datetime import date, timedelta

import faker

## 1 - Preparing the Data

In [11]:
# create the faker to populate the data
fake = faker.Faker()

In [12]:
# generate user profiles
# simulate user data coming from an API. It is a list
# of JSON strings (users).
def get_users(no_of_users):
    usernames = (
        fake.unique.user_name() for i in range(no_of_users)
    )
    genders = random.choices(
        ["M", "F", "O"], weights=[0.43, 0.47, 0.1], k=no_of_users
    )
    for username, gender in zip(usernames, genders):
        user = {
            "username": username,
            "name": get_random_name(gender),
            "gender": gender,
            "email": fake.email(),
            "age": fake.random_int(min=18, max=90),
            "address": fake.address(),
        }
        yield json.dumps(user)


def get_random_name(gender):
    match gender:
        case "F":
            name = fake.name_female()
        case "M":
            name = fake.name_male()
        case _:
            name = fake.name_nonbinary()
    return name


users = list(get_users(1000))
users[:3]

['{"username": "kelly60", "name": "Lori Yates", "gender": "F", "email": "paulford@example.com", "age": 51, "address": "573 Hunter Corner Apt. 129\\nNew Jamie, WY 27197"}',
 '{"username": "umontgomery", "name": "Craig Johnson", "gender": "M", "email": "lewistanya@example.net", "age": 76, "address": "920 Rachel Drive Apt. 601\\nEast Ronaldton, WA 54856"}',
 '{"username": "hcoleman", "name": "Denise Garcia", "gender": "F", "email": "grantbethany@example.com", "age": 41, "address": "620 Patterson Hollow Suite 577\\nChristopherstad, LA 04494"}']

In [13]:
# campaign name format:
# InternalType_StartDate_EndDate_TargetAge_TargetGender_Currency
def get_type():
    # just some meaningless example codes
    types = ["AKX", "BYU", "GRZ", "KTR"]
    return random.choice(types)


def get_start_end_dates():
    duration = random.randint(1, 2 * 365)
    offset = random.randint(-365, 365)
    start = date.today() - timedelta(days=offset)
    end = start + timedelta(days=duration)

    def _format_date(date_):
        return date_.strftime("%Y%m%d")

    return _format_date(start), _format_date(end)


def get_age_range():
    age = random.randrange(20, 46, 5)
    diff = random.randrange(5, 26, 5)
    return "{}-{}".format(age, age + diff)


def get_gender():
    return random.choice(("M", "F", "A"))


def get_currency():
    return random.choice(("GBP", "EUR", "USD"))


def get_campaign_name():
    separator = "_"
    type_ = get_type()
    start, end = get_start_end_dates()
    age_range = get_age_range()
    gender = get_gender()
    currency = get_currency()
    return separator.join(
        (type_, start, end, age_range, gender, currency)
    )

In [14]:
# campaign data:
# name, budget, spent, clicks, impressions
def get_campaign_data():
    name = get_campaign_name()
    budget = random.randint(10**3, 10**6)
    spent = random.randint(10**2, budget)
    clicks = int(random.triangular(10**2, 10**5, 0.2 * 10**5))
    impressions = int(random.gauss(0.5 * 10**6, 2))
    return {
        "cmp_name": name,
        "cmp_bgt": budget,
        "cmp_spent": spent,
        "cmp_clicks": clicks,
        "cmp_impr": impressions,
    }

In [15]:
# assemble the logic to get the final version of the rough data
# data will be a list of dictionaries. Each dictionary will follow
# this structure:
# {'user': user_json, 'campaigns': [c1, c2, ...]}
# where user_json is the JSON string version of a user data dict
# and c1, c2, ... are campaign dicts as returned by
# get_campaign_data


def get_data(users):
    data = []
    for user in users:
        campaigns = [
            get_campaign_data()
            for _ in range(random.randint(2, 8))
        ]
        data.append({"user": user, "campaigns": campaigns})
    return data

## 2 - Cleaning the data

In [16]:
# fetch simulated rough data
rough_data = get_data(users)

rough_data[:2]  # let us take a peek

[{'user': '{"username": "kelly60", "name": "Lori Yates", "gender": "F", "email": "paulford@example.com", "age": 51, "address": "573 Hunter Corner Apt. 129\\nNew Jamie, WY 27197"}',
  'campaigns': [{'cmp_name': 'AKX_20260327_20270725_35-45_M_USD',
    'cmp_bgt': 242195,
    'cmp_spent': 202021,
    'cmp_clicks': 22564,
    'cmp_impr': 500002},
   {'cmp_name': 'BYU_20250506_20251114_25-50_M_GBP',
    'cmp_bgt': 682646,
    'cmp_spent': 208570,
    'cmp_clicks': 66373,
    'cmp_impr': 500002},
   {'cmp_name': 'KTR_20270409_20271102_30-50_M_GBP',
    'cmp_bgt': 518803,
    'cmp_spent': 12763,
    'cmp_clicks': 49996,
    'cmp_impr': 500001},
   {'cmp_name': 'GRZ_20260208_20260620_40-55_F_EUR',
    'cmp_bgt': 26193,
    'cmp_spent': 6330,
    'cmp_clicks': 61776,
    'cmp_impr': 500000},
   {'cmp_name': 'KTR_20260930_20271015_25-45_A_GBP',
    'cmp_bgt': 997541,
    'cmp_spent': 494333,
    'cmp_clicks': 41566,
    'cmp_impr': 500000}]},
 {'user': '{"username": "umontgomery", "name": "Craig

In [17]:
# Let's start from having a different version of the data
# We want a list whose items will be dicts. Each dict is
# the original campaign dict plus the user JSON

data = []
for datum in rough_data:
    for campaign in datum["campaigns"]:
        campaign.update({"user": datum["user"]})
        data.append(campaign)
data[:2]  # let us take another peek

[{'cmp_name': 'AKX_20260327_20270725_35-45_M_USD',
  'cmp_bgt': 242195,
  'cmp_spent': 202021,
  'cmp_clicks': 22564,
  'cmp_impr': 500002,
  'user': '{"username": "kelly60", "name": "Lori Yates", "gender": "F", "email": "paulford@example.com", "age": 51, "address": "573 Hunter Corner Apt. 129\\nNew Jamie, WY 27197"}'},
 {'cmp_name': 'BYU_20250506_20251114_25-50_M_GBP',
  'cmp_bgt': 682646,
  'cmp_spent': 208570,
  'cmp_clicks': 66373,
  'cmp_impr': 500002,
  'user': '{"username": "kelly60", "name": "Lori Yates", "gender": "F", "email": "paulford@example.com", "age": 51, "address": "573 Hunter Corner Apt. 129\\nNew Jamie, WY 27197"}'}]

In [18]:
# Warning: Uncommenting and executing this cell will overwrite data.json
# with open("data.json", "w") as stream:
#     stream.write(json.dumps(data))